In [1]:
# -*- coding: utf-8 -*-
"""
train_and_predict_v6_ultrafast.ipynb
------------------------------------
单元格1：模型训练与评估
"""

import re
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PowerTransformer, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, r2_score

RANDOM_STATE = 111
np.random.seed(RANDOM_STATE)

def extract_number(x):
    if pd.isna(x): return np.nan
    m = re.search(r"(-?\d+\.?\d*)", str(x))
    return float(m.group(1)) if m else np.nan

def parse_floor_ratio(s):
    s = str(s)
    nums = re.findall(r"(\d+\.?\d*)", s)
    if len(nums) >= 2:
        return float(nums[0]) / float(nums[-1]) if float(nums[-1]) != 0 else np.nan
    return float(nums[0]) if len(nums) == 1 else np.nan

def extract_year(s):
    s = str(s)
    m = re.search(r"(19|20)\d{2}", s)
    return int(m.group(0)) if m else np.nan

def iqr_clip(sr):
    q1, q3 = np.nanpercentile(sr, [25, 75])
    iqr = q3 - q1
    return np.clip(sr, q1 - 1.5 * iqr, q3 + 1.5 * iqr)

def clean_train(df, task):
    drop_cols = ["开发商","物业公司","物业办公电话","产权描述","客户反馈",
                 "coord_x","coord_y","房屋优势","核心卖点","户型介绍","周边配套","交通出行"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    for c in ["面积","建筑面积","套内面积"]:
        if c in df.columns: df[c] = df[c].apply(extract_number)
    if "所在楼层" in df.columns: df["楼层比例"] = df["所在楼层"].apply(parse_floor_ratio)
    if "建筑年代" in df.columns:
        df["建筑年代"] = df["建筑年代"].apply(extract_year)
        df["房龄"] = 2025 - df["建筑年代"]
    if {"lon","lat"}.issubset(df.columns):
        df["lon_std"] = (df["lon"] - df["lon"].mean()) / (df["lon"].std() or 1)
        df["lat_std"] = (df["lat"] - df["lat"].mean()) / (df["lat"].std() or 1)
        df["lon_lat_interaction"] = df["lon_std"] * df["lat_std"]
    if {"面积","房龄"}.issubset(df.columns):
        df["面积房龄交互"] = np.log1p(df["面积"]) * df["房龄"]

    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
    if df["Price"].median() < 1000: df["Price"] *= 10000
    df["Price"] = iqr_clip(df["Price"])

    cat_cols = [c for c in ["城市","区域","区县","板块","装修","朝向","物业类别","建筑结构",
                            "租赁方式","付款方式","电梯","车位","环线位置"] if c in df.columns]
    for c in cat_cols: df[c] = df[c].astype(str).fillna("未知")
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    df = df.fillna(df.median(numeric_only=True)).fillna(0)
    return df

def train_models(df, task):
    y = df["Price"].values
    X = df.drop(columns=["Price"], errors="ignore").select_dtypes(include=[np.number])
    pt_y = PowerTransformer(method='yeo-johnson')
    y_trans = pt_y.fit_transform(y.reshape(-1, 1)).flatten()
    joblib.dump(pt_y, f"target_transformer_{task}.pkl")
    X_train, X_test, y_train, y_test = train_test_split(X, y_trans, test_size=0.2, random_state=RANDOM_STATE)
    joblib.dump(X.columns.tolist(), f"model_features_{task}.pkl")

    core_poly = [c for c in ["面积","房龄","楼层比例","lon_std","lat_std"] if c in X.columns]
    preproc = ColumnTransformer([("poly", PolynomialFeatures(degree=2, include_bias=False), core_poly),
                                 ("pass", "passthrough", X.columns)], remainder='drop')

    models = {
        "OLS": LinearRegression(),
        "Ridge": Ridge(random_state=RANDOM_STATE),
        "LASSO": Lasso(max_iter=20000, random_state=RANDOM_STATE),
        "ElasticNet": ElasticNet(max_iter=20000, random_state=RANDOM_STATE)
    }
    params = {
        "Ridge": {"model__alpha": np.logspace(-2, 2, 6)},
        "LASSO": {"model__alpha": np.logspace(-3, 1, 5)},
        "ElasticNet": {"model__alpha": np.logspace(-3, 1, 5), "model__l1_ratio": [0.3,0.5,0.7]}
    }

    inv = lambda arr: pt_y.inverse_transform(arr.reshape(-1,1)).flatten()
    cv = KFold(n_splits=6, shuffle=True, random_state=RANDOM_STATE)
    results = []

    for name, est in models.items():
        pipe = Pipeline([("prep", preproc), ("scaler", StandardScaler()), ("model", est)])
        model = (GridSearchCV(pipe, params[name], cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1).fit(X_train, y_train)
                 .best_estimator_ if name in params else pipe.fit(X_train, y_train))
        y_pred = inv(model.predict(X_test))
        y_true = inv(y_test)
        results.append({"Model": name,
                        "MAE": mean_absolute_error(y_true, y_pred),
                        "R²": r2_score(y_true, y_pred)})

        plt.figure(figsize=(5,5))
        plt.scatter(y_true, y_pred, s=10, alpha=0.5)
        lo, hi = y_true.min(), y_true.max()
        plt.plot([lo, hi], [lo, hi], "r--")
        plt.title(f"{task.upper()} - {name}")
        plt.savefig(f"scatter_{task}_{name}.png", dpi=150)
        plt.close()

        joblib.dump(model, f"model_{task}_{name}.pkl")

    pd.DataFrame(results).to_csv(f"performance_summary_{task}.csv", index=False)

price_df = pd.read_excel("ruc_Class25Q2_train_price.xlsx")
rent_df  = pd.read_excel("ruc_Class25Q2_train_rent.xlsx")
train_models(clean_train(price_df, "price"), "price")
train_models(clean_train(rent_df, "rent"), "rent")



In [2]:
# -*- coding: utf-8 -*-
"""
train_and_predict_v6_ultrafast.ipynb
------------------------------------
单元格2：预测与结果导出
"""

import numpy as np
import pandas as pd
import joblib
import re

def extract_number(x):
    if pd.isna(x): return np.nan
    m = re.search(r"(-?\d+\.?\d*)", str(x))
    return float(m.group(1)) if m else np.nan

def parse_floor_ratio(s):
    s = str(s)
    nums = re.findall(r"(\d+\.?\d*)", s)
    if len(nums) >= 2:
        return float(nums[0]) / float(nums[-1]) if float(nums[-1]) != 0 else np.nan
    return float(nums[0]) if len(nums) == 1 else np.nan

def extract_year(s):
    s = str(s)
    m = re.search(r"(19|20)\d{2}", s)
    return int(m.group(0)) if m else np.nan

def clean_test(df_raw):
    df = df_raw.copy()
    for c in ["面积","建筑面积","套内面积"]:
        if c in df.columns: df[c] = df[c].apply(extract_number)
    if "所在楼层" in df.columns: df["楼层比例"] = df["所在楼层"].apply(parse_floor_ratio)
    if "建筑年代" in df.columns:
        df["建筑年代"] = df["建筑年代"].apply(extract_year)
        df["房龄"] = 2025 - df["建筑年代"]
    if {"lon","lat"}.issubset(df.columns):
        df["lon_std"] = (df["lon"] - df["lon"].mean()) / (df["lon"].std() or 1)
        df["lat_std"] = (df["lat"] - df["lat"].mean()) / (df["lat"].std() or 1)
        df["lon_lat_interaction"] = df["lon_std"] * df["lat_std"]
    if {"面积","房龄"}.issubset(df.columns):
        df["面积房龄交互"] = np.log1p(df["面积"]) * df["房龄"]
    cat_cols = [c for c in ["城市","区域","区县","板块","装修","朝向",
                            "物业类别","建筑结构","租赁方式","付款方式",
                            "电梯","车位","环线位置"] if c in df.columns]
    for c in cat_cols: df[c] = df[c].astype(str).fillna("未知")
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    df = df.fillna(df.median(numeric_only=True)).fillna(0)
    return df

def stabilize_predictions(pred):
    pred = np.clip(pred, 0, None)
    q1, q3 = np.percentile(pred, [10, 90])
    iqr = q3 - q1
    return np.clip(pred, q1 - 0.5 * iqr, q3 + 1.5 * iqr)

def predict_and_merge(model_name, df_price, df_rent):
    feat_price = joblib.load("model_features_price.pkl")
    feat_rent  = joblib.load("model_features_rent.pkl")
    pt_price   = joblib.load("target_transformer_price.pkl")
    pt_rent    = joblib.load("target_transformer_rent.pkl")
    mdl_price  = joblib.load(f"model_price_{model_name}.pkl")
    mdl_rent   = joblib.load(f"model_rent_{model_name}.pkl")

    Xp = df_price.reindex(columns=feat_price, fill_value=0)
    Xr = df_rent.reindex(columns=feat_rent, fill_value=0)
    yp = pt_price.inverse_transform(mdl_price.predict(Xp).reshape(-1,1)).flatten()
    yr = pt_rent.inverse_transform(mdl_rent.predict(Xr).reshape(-1,1)).flatten()
    yp, yr = stabilize_predictions(yp), stabilize_predictions(yr)

    sub = pd.concat([
        pd.DataFrame({"ID": df_price["ID"], "Price": yp}),
        pd.DataFrame({"ID": df_rent["ID"], "Price": yr})
    ]).sort_values("ID")

    sub.to_csv(f"submission_{model_name}.csv", index=False, encoding="utf-8-sig")
    print(f"✅ 导出 submission_{model_name}.csv，共 {len(sub)} 条")

df_price = pd.read_excel("ruc_Class25Q2_test_price.xlsx")
df_rent  = pd.read_excel("ruc_Class25Q2_test_rent.xlsx")
df_price_t = clean_test(df_price)
df_rent_t  = clean_test(df_rent)

for model in ["OLS","Ridge","LASSO","ElasticNet"]:
    predict_and_merge(model, df_price_t, df_rent_t)


✅ 导出 submission_OLS.csv，共 43790 条
✅ 导出 submission_Ridge.csv，共 43790 条
✅ 导出 submission_LASSO.csv，共 43790 条
✅ 导出 submission_ElasticNet.csv，共 43790 条
